# 🎯 WikiQuiz — Challenge 1
Interactive quiz generation from Wikimedia Structured Wikipedia content.

Each code cell is preceded by a clear heading explaining what it performs.


# 1. Install Required Dataset Library


In [ ]:
!pip install -q kagglehub[pandas-datasets]


# 2. Import Libraries for Dataset Loading


In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter


# 3. Load the Wikipedia Structured Contents Dataset


In [ ]:
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "wikimedia-foundation/wikipedia-structured-contents",
    "enwiki/data/enwiki_namespace_0_00008.parquet"
)
print("Dataset shape:", df.shape)
print(df.columns.tolist())


# 4. Search for a Wikipedia Topic


In [ ]:
topic = input("Enter a Wikipedia topic: ")
results = df[df['name'].astype(str).str.contains(topic, case=False, na=False)]
print(results[['name', 'abstract']].head(5))


# 5. Select an Article and Extract Its Information


In [ ]:
if results.empty:
    print("No matching article found. Try another topic.")
else:
    article = results.iloc[0]
    print("Topic:", article['name'])
    print("Description:", article.get('description', ''))
    print("Abstract:", article.get('abstract', ''))


# 6. Define the Rule-Based Question Generator
Questions are generated from information available in the selected article and dataset.


In [ ]:
import re
import random

def clean(value):
    if pd.isna(value):
        return ""
    return str(value).strip()

def generate_questions(article, dataset, max_questions=5):
    title = clean(article.get('name'))
    description = clean(article.get('description'))
    abstract = clean(article.get('abstract'))
    text = ' '.join(x for x in [description, abstract] if x)
    questions = []

    if title:
        questions.append({
            'question': 'What is the title of the selected article?',
            'options': [title] + [clean(x) for x in dataset['name'].dropna().sample(min(3, len(dataset))).tolist() if clean(x) != title],
            'answer': title,
            'explanation': f'The selected article is {title}.'
        })

    if description:
        candidates = [clean(x) for x in dataset['description'].dropna().tolist() if clean(x) and clean(x) != description]
        random.shuffle(candidates)
        opts = [description] + candidates[:3]
        if len(opts) == 4:
            questions.append({
                'question': 'Which description matches the selected article?',
                'options': opts, 'answer': description,
                'explanation': 'This description is taken directly from the selected dataset record.'
            })

    years = re.findall(r'\b(?:18|19|20)\d{2}\b', text)
    if years:
        year = years[0]
        year_candidates = list(dict.fromkeys(re.findall(r'\b(?:18|19|20)\d{2}\b', ' '.join(dataset['abstract'].dropna().astype(str).head(500)))))
        year_candidates = [y for y in year_candidates if y != year]
        opts = [year] + year_candidates[:3]
        if len(opts) == 4:
            questions.append({'question':'Which year is mentioned in the article?','options':opts,'answer':year,'explanation':f'The year {year} appears in the article information.'})

    lower = text.lower()
    sports = ['football','basketball','cricket','tennis','hockey','baseball','volleyball','swimming']
    found_sport = next((s for s in sports if s in lower), None)
    if found_sport:
        questions.append({'question':'Which sport is mentioned in the article?','options':random.sample([found_sport] + [s for s in sports if s != found_sport], 4),'answer':found_sport,'explanation':f'The article mentions {found_sport}.'})

    if text:
        questions.append({'question':'True or False: The selected article contains information in its abstract or description.','options':['True','False'],'answer':'True','explanation':'The quiz was generated from the article information available in the dataset.'})

    random.shuffle(questions)
    return questions[:max_questions]


# 7. Run the Quiz and Calculate the Final Score


In [ ]:
questions = generate_questions(article, df)
score = 0

for i, q in enumerate(questions, 1):
    print(f'\nQuestion {i}: {q["question"]}')
    for j, option in enumerate(q['options'], 1):
        print(f'{j}. {option}')
    choice = int(input('Your answer: ')) - 1
    selected = q['options'][choice]
    if selected == q['answer']:
        score += 1
        print('✓ Correct!')
    else:
        print(f'✗ Incorrect. Correct answer: {q["answer"]}')
    print('Explanation:', q['explanation'])

print(f'\nFinal Score: {score} / {len(questions)}')
if questions:
    print(f'Percentage: {score / len(questions) * 100:.0f}%')
